# Cube Bot (Cubli) — Inverted Pendulum Control

Python conversion of the MATLAB Live Scripts (`final_project.mlx`, `topic_8.mlx`).

A planar **cube bot / Cubli** balances upright using a **reaction wheel**. The body angle $\theta_b$ is the tilt from vertical; the wheel angle/speed $\theta_w$ produces counter-torque.

## Controllers used in this project

| Controller | Where applied | Gains / design |
|---|---|---|
| **PID** | Linear and nonlinear plant models | $K_p=20$, $K_i=10$, $K_d=15$ on body angle $\theta_b$ |
| **LQR** | State-feedback on linearized dynamics | Continuous LQR with $Q$, $R$ weighting matrices |

A brief **disturbance / load** pulse is injected around $t \in [15, 16]$ s to test recovery.

Reference geometry: *The Cubli: A cube that can jump up and balance* (Gajamohan, Merz, Thommen, D'Andrea).

## 1. Imports and physical parameters

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.linalg import solve_continuous_are
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Physical parameters (same as MATLAB)
lb = 0.075   # m  — CoM of body to pivot
l  = 0.085   # m  — wheel axis to pivot
Mb = 0.419   # kg — body mass
Mw = 0.204   # kg — wheel mass
Ib = 3.34e-3 # kg.m^2 — body inertia about pivot
Iw = 0.57e-3 # kg.m^2 — wheel + rotor inertia
Cb = 1.02e-3 # kg.m^2/s — body friction
Cw = 0.05e-3 # kg.m^2/s — wheel friction
g  = 9.81    # m/s^2

print("Parameters loaded.")

## 2. Mathematical modeling (from MATLAB `topic_8.mlx`)

Planar cube bot (Cubli) for an inverted-pendulum balancing task.
Size: $15\times 15\,\mathrm{cm}^2$.

**Source figure:** *The Cubli: A cube that can jump up and balance* (Gajamohan, Merz, Thommen, D'Andrea).


In [ ]:
from IPython.display import Image, display
display(Image("WhatsApp Image 2023-11-23 at 00.47.14_b943f96e.jpg", width=640))
print("Fig. 1 — Planar cube bot geometry (project schematic).")


### Symbols

| Symbol | Meaning |
|---|---|
| $l_b$ | Distance from body CoM (frame + fasteners, etc.) to the pivot |
| $l$ | Distance from wheel axis to the pivot |
| $\theta_b$ | Angle between the body diagonal (through the pivot) and the vertical |
| $\theta_w$ | Wheel rotation relative to the body |
| $I_b$ | Body moment of inertia about the pivot axis (⊥ to the plane) |
| $I_w$ | Wheel + motor-rotor inertia about the motor axis |
| $C_b$ | Dynamic friction coefficient of the body |
| $C_w$ | Dynamic friction coefficient of the wheel |
| $T_m$ | Motor torque |
| $m_b, m_w$ | Body and wheel masses (denoted $M_b, M_w$ in the code) |


### Kinetic energy

Body:

$$
T_b = \frac{1}{2} I_b \dot{\theta}_b^2
$$

Wheel (translation of the hub at distance $l$, plus spin about its axis):

$$
T_w = \frac{1}{2} m_w \bigl(l\,\dot{\theta}_b\bigr)^2 + \frac{1}{2} I_w \bigl(\dot{\theta}_b + \dot{\theta}_w\bigr)^2
$$

Total:

$$
T = T_b + T_w
= \frac{1}{2} I_b \dot{\theta}_b^2
+ \frac{1}{2} m_w \bigl(l\,\dot{\theta}_b\bigr)^2
+ \frac{1}{2} I_w \bigl(\dot{\theta}_b + \dot{\theta}_w\bigr)^2
$$

### Potential energy

$$
V_b = m_b\, g\, l_b\, \cos(\theta_b),
\qquad
V_w = m_w\, g\, l\, \cos(\theta_b)
$$

$$
V = V_b + V_w = (m_b l_b + m_w l)\, g\, \cos(\theta_b)
$$

### Lagrangian

$$
L = T - V
= \frac{1}{2} I_b \dot{\theta}_b^2
+ \frac{1}{2} m_w (l\dot{\theta}_b)^2
+ \frac{1}{2} I_w (\dot{\theta}_b + \dot{\theta}_w)^2
- (m_b l_b + m_w l)\, g\, \cos(\theta_b)
$$


### Euler–Lagrange equations (with friction / motor torque)

For the body coordinate $\theta_b$ (viscous friction $-C_b\dot{\theta}_b$):

$$
\frac{d}{dt}\!\left(\frac{\partial L}{\partial \dot{\theta}_b}\right)
- \frac{\partial L}{\partial \theta_b}
= - C_b \dot{\theta}_b
$$

$$
\frac{d}{dt}\Bigl(
I_b\dot{\theta}_b + m_w l^2\dot{\theta}_b + I_w(\dot{\theta}_b + \dot{\theta}_w)
\Bigr)
- (m_b l_b + m_w l)\, g\, \sin(\theta_b)
= - C_b \dot{\theta}_b
$$

$$
I_b\ddot{\theta}_b + m_w l^2\ddot{\theta}_b + I_w(\ddot{\theta}_b + \ddot{\theta}_w)
- (m_b l_b + m_w l)\, g\, \sin(\theta_b)
= - C_b \dot{\theta}_b
$$

For the wheel coordinate $\theta_w$ (motor torque $T_m$, friction $-C_w\dot{\theta}_w$):

$$
\frac{d}{dt}\!\left(\frac{\partial L}{\partial \dot{\theta}_w}\right)
- \frac{\partial L}{\partial \theta_w}
= T_m - C_w \dot{\theta}_w
$$

$$
\frac{d}{dt}\bigl( I_w(\dot{\theta}_b + \dot{\theta}_w) \bigr)
= T_m - C_w \dot{\theta}_w
$$

$$
I_w(\ddot{\theta}_b + \ddot{\theta}_w) = T_m - C_w \dot{\theta}_w
$$

### Solved accelerations (nonlinear EOM)

$$
\ddot{\theta}_b
=
\frac{
(m_b l_b + m_w l)\, g\, \sin(\theta_b)
- T_m
+ C_w\dot{\theta}_w
- C_b\dot{\theta}_b
}{I_b + m_w l^2}
$$

$$
\ddot{\theta}_w
=
\frac{T_m - C_w\dot{\theta}_w}{I_w}
-
\frac{
(m_b l_b + m_w l)\, g\, \sin(\theta_b)
- T_m
+ C_w\dot{\theta}_w
- C_b\dot{\theta}_b
}{I_b + m_w l^2}
$$

Motor model used in the notes:

$$
T_m = K_m i,
\qquad
K_m = 25.1\times 10^{-3}\,\mathrm{N{\cdot}m{\cdot}A^{-1}}
$$
(brushless DC torque constant).


### State-space form (3-state model from `topic_8`)

$$
\theta_b = x_1,\quad
\dot{\theta}_b = x_2,\ \ddot{\theta}_b=\dot{x}_2,\quad
\dot{\theta}_w = x_3,\ \ddot{\theta}_w=\dot{x}_3
$$

$$
\begin{aligned}
\dot{x}_1 &= x_2 \\[4pt]
\dot{x}_2 &=
\frac{(m_b l_b + m_w l)\, g\, \sin(x_1) - T_m + C_w x_3 - C_b x_2}{I_b + m_w l^2} \\[4pt]
\dot{x}_3 &=
\frac{T_m - C_w x_3}{I_w}
-
\frac{(m_b l_b + m_w l)\, g\, \sin(x_1) - T_m + C_w x_3 - C_b x_2}{I_b + m_w l^2}
\end{aligned}
$$

### Linearization at the upright equilibrium $(0,0,0)$

With $\sin(x_1)\approx x_1$:

$$
\begin{pmatrix}\dot{x}_1\\ \dot{x}_2\\ \dot{x}_3\end{pmatrix}
=
\begin{pmatrix}
0 & 1 & 0 \\[4pt]
\dfrac{(m_b l_b + m_w l)g}{I_b + m_w l^2}
& -\dfrac{C_b}{I_b + m_w l^2}
& \dfrac{C_w}{I_b + m_w l^2} \\[10pt]
-\dfrac{(m_b l_b + m_w l)g}{I_b + m_w l^2}
& \dfrac{C_b}{I_b + m_w l^2}
& -\dfrac{C_w(I_w + I_b + m_w l^2)}{I_w(I_b + m_w l^2)}
\end{pmatrix}
\begin{pmatrix}x_1\\ x_2\\ x_3\end{pmatrix}
+
\begin{pmatrix}
0 \\[4pt]
-\dfrac{1}{I_b + m_w l^2} \\[10pt]
\dfrac{I_w + I_b + m_w l^2}{I_w(I_b + m_w l^2)}
\end{pmatrix}
T_m
$$

> **Note:** `final_project.mlx` uses a related **4-state** model
> $X=[\theta_b,\theta_w,\dot{\theta}_b,\dot{\theta}_w]^\top$ for simulation/LQR;
> the derivation above is the 3-state form written out in `topic_8.mlx`.
> Both share the same Lagrangian physics.


## 3. Dynamics and controllers

State for the **4-state** model used in `final_project.mlx`:

$$X = [\theta_b,\; \theta_w,\; \dot\theta_b,\; \dot\theta_w]^T$$

Nonlinear equations of motion (Lagrangian model):

$$\ddot\theta_b = \frac{(M_b l_b + M_w l) g \sin\theta_b - T_m - C_b\dot\theta_b + C_w\dot\theta_w}{I_b + M_w l^2}$$

$$\ddot\theta_w = \frac{(I_b+I_w+M_w l^2)(T_m - C_w\dot\theta_w)}{I_w(I_b+M_w l^2)} - \frac{(M_b l_b + M_w l)g\sin\theta_b - C_b\dot\theta_b}{I_b+M_w l^2}$$

In [ ]:
class PIDState:
    """Mutable integral state for the PID (replaces MATLAB globals)."""
    def __init__(self):
        self.integral = 0.0
        self.tp = 0.0

    def reset(self):
        self.integral = 0.0
        self.tp = 0.0


pid = PIDState()
K_lqr = None  # filled after LQR design


def Load(t):
    """Velocity disturbance pulse (final_project.mlx)."""
    return 0.1 if 15.0 < t < 16.0 else 0.0


def disturbance_angle(t):
    """Angle disturbance pulse (topic_8.mlx)."""
    return np.pi / 8 if 15.0 < t < 16.0 else 0.0


def pid_control(t, X, use_4state=True):
    """PID on body angle: T = Kp*e + Ki*∫e + Kd*ė

    Gains from MATLAB: kp=20, ki=10, kd=15, setpoint=0.
    """
    kp, ki, kd = 20.0, 10.0, 15.0
    e = X[0] - 0.0
    de = X[2] if use_4state else X[1]  # body angular velocity
    dt = t - pid.tp
    if dt > 0:
        pid.integral += ki * e * dt
    pid.tp = t
    return kp * e + pid.integral + kd * de


def accelerations(theta_b, wb, ww, Torque, nonlinear=True):
    """Return (theta_b_ddot, theta_w_ddot)."""
    denom = Ib + Mw * l * l
    gravity = (Mb * lb + Mw * l) * g * (np.sin(theta_b) if nonlinear else theta_b)
    ddb = (gravity - Torque - Cb * wb + Cw * ww) / denom
    ddw = ((Ib + Iw + Mw * l * l) * (Torque - Cw * ww)) / (Iw * denom) - (gravity - Cb * wb) / denom
    return ddb, ddw


def sysdyn(t, X):
    """Nonlinear plant + PID (4-state)."""
    load = Load(t)
    x1, x2, x3, x4 = X[0], X[1], X[2] + load, X[3]
    Torque = pid_control(t, X, use_4state=True)
    ddb, ddw = accelerations(x1, x3, x4, Torque, nonlinear=True)
    return [x3, x4, ddb, ddw]


def lin_sysdyn(t, X):
    """Linearized plant + PID (4-state)."""
    load = Load(t)
    x1, x2, x3, x4 = X[0], X[1], X[2] + load, X[3]
    Torque = pid_control(t, X, use_4state=True)
    ddb, ddw = accelerations(x1, x3, x4, Torque, nonlinear=False)
    return [x3, x4, ddb, ddw]


def sysdyn_lqr(t, X):
    """Linearized plant + LQR state feedback (4-state)."""
    load = Load(t)
    x1, x2, x3, x4 = X[0], X[1], X[2] + load, X[3]
    Torque = float(-K_lqr @ X)
    ddb, ddw = accelerations(x1, x3, x4, Torque, nonlinear=False)
    return [x3, x4, ddb, ddw]


def simulate(fun, t_span, x0, n_eval=2000):
    pid.reset()
    t_eval = np.linspace(t_span[0], t_span[1], n_eval)
    sol = solve_ivp(fun, t_span, x0, t_eval=t_eval, rtol=1e-6, atol=1e-8, method="RK45")
    return sol.t, sol.y.T


def plot_states(t, X, title, angle_in_deg=True):
    fig, axs = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
    fig.suptitle(title)
    angle = X[:, 0] * (180 / np.pi if angle_in_deg else 1.0)
    axs[0].plot(t, angle)
    axs[0].set_ylabel(r"$\theta_b$" + (" (deg)" if angle_in_deg else " (rad)"))
    axs[0].axhline(0, color="k", lw=0.5)
    axs[1].plot(t, X[:, 2])
    axs[1].set_ylabel(r"$\dot\theta_b$ (rad/s)")
    axs[2].plot(t, X[:, 3])
    axs[2].set_ylabel(r"$\dot\theta_w$ (rad/s)")
    axs[2].set_xlabel("time (s)")
    for ax in axs:
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print("Dynamics and controllers defined.")

## 4. PID on linearized system

Initial tilt $\theta_b(0) = \pi/18$ (10°). PID tries to regulate $\theta_b \to 0$.

In [ ]:
t_span = (0.0, 40.0)
X0 = np.array([np.pi / 18, 0.0, 0.0, 0.0])

t_lin, X_lin = simulate(lin_sysdyn, t_span, X0)
plot_states(t_lin, X_lin, "PID — linearized plant")

## 5. PID on nonlinear system

In [ ]:
t_nl, X_nl = simulate(sysdyn, t_span, X0)
plot_states(t_nl, X_nl, "PID — nonlinear plant")

## 6. LQR state-feedback controller

Classic continuous LQR for the linearized 4-state model about the **upright** equilibrium ($\theta_b=0$):

$$\dot X = A X + B T_m,\qquad u = -K X,\qquad J = \int (X^T Q X + R u^2)\,dt$$

Desired state is fixed at upright: $X_{des}=0$. MATLAB weights from `final_project.mlx`:

$$Q = \mathrm{diag}(10,\,1,\,10,\,0.1),\quad R = 500$$


In [ ]:
denom = Ib + Mw * l * l
A = np.array([
    [0, 0, 1, 0],
    [0, 0, 0, 1],
    [(Mb * lb + Mw * l) * g / denom, 0, -Cb / denom, Cw / denom],
    [-(Mb * lb + Mw * l) * g / denom, 0, Cb / denom, (Ib + Iw + Mw * l * l) * (-Cw) / (Iw * denom)],
])
B = np.array([
    [0.0],
    [0.0],
    [-1.0 / denom],
    [(Ib + Iw + Mw * l * l) / (Iw * denom)],
])

Q = np.diag([10.0, 1.0, 10.0, 0.1])
R = np.array([[500.0]])

# Continuous Algebraic Riccati Equation → LQR gain K = R^{-1} B^T P
P = solve_continuous_are(A, B, Q, R)
K_lqr = (np.linalg.solve(R, B.T @ P)).ravel()
print("LQR gain K =", K_lqr)


def sysdyn_lqr(t, X):
    """Linearized plant + upright LQR: u = -K X (θ_des = 0)."""
    load = Load(t)
    x1, x3, x4 = X[0], X[2] + load, X[3]
    Torque = float(-K_lqr @ X)
    ddb, ddw = accelerations(x1, x3, x4, Torque, nonlinear=False)
    return [x3, x4, ddb, ddw]


X0_lqr = np.array([np.pi / 8, 0.0, 0.0, 0.0])  # 22.5° initial tilt
t_lqr, X_lqr = simulate(sysdyn_lqr, t_span, X0_lqr)
plot_states(t_lqr, X_lqr, "LQR — upright (θ_des = 0)")

T_hist = np.array([-float(K_lqr @ x) for x in X_lqr])
plt.figure(figsize=(9, 3))
plt.plot(t_lqr, T_hist)
plt.xlabel("time (s)")
plt.ylabel(r"$T_m$ (N·m)")
plt.title("LQR motor torque")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Cubli animation
 the bot initially tilted by $\theta_w$.


In [ ]:
def animate_cubli(
    t,
    X,
    title="Cubli — reaction-wheel balancer",
    n_frames=160,
    interval=35,
    show_labels=True,
    trail_ghost=True,
    theta_des=0.0,
):
    """Realistic planar Cubli animation matching the project schematic.

    Geometry (body frame, pivot at origin, upright diagonal along +y):
      - 15 cm square resting on a corner
      - reaction wheel centre at distance l along the diagonal
      - body CoM at distance lb along the diagonal
    State X columns: [theta_b, theta_w, theta_b_dot, theta_w_dot]
    """
    from matplotlib.patches import Circle

    # Size the square so the diagonal intersection (geometric centre)
    # lies at distance l from the pivot — that is where the reaction
    # wheel axis sits. Then lb stays on the same diagonal (lb < l).
    #   centre distance = side/√2  ⇒  side = l√2,  half = l,  diag = 2l
    half = l
    side = l * np.sqrt(2)
    diag = 2.0 * l
    r_wheel = 0.55 * half  # rim fits inside the square about the centre

    # Square vertices: bottom corner = pivot (0,0), upright = diamond
    # Diagonals: (0,0)→(0,diag) and (-half,half)→(half,half) meet at (0,l)
    verts0 = np.array([
        [0.0, 0.0],
        [half, half],
        [0.0, diag],
        [-half, half],
    ])

    def rot(ang):
        c, s = np.cos(ang), np.sin(ang)
        return np.array([[c, -s], [s, c]])

    def body_to_world(pts, theta_b):
        return pts @ rot(theta_b).T

    idx = np.linspace(0, len(t) - 1, n_frames).astype(int)
    th_b = X[idx, 0]
    th_w = X[idx, 1]
    wb = X[idx, 2]
    ww = X[idx, 3]

    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    fig.patch.set_facecolor("#f7f7f5")
    ax.set_facecolor("#f7f7f5")

    # Ground / floor (schematic base line)
    ax.plot([-0.28, 0.32], [0.0, 0.0], color="#222", lw=1.8, zorder=1)
    ax.fill_between([-0.32, 0.36], -0.045, 0.0, color="#ddd8d0", zorder=0)

    # Vertical reference (upright equilibrium)
    ax.plot([0, 0], [0, diag + 0.02], color="#999", lw=1.0, ls="--", zorder=1)

    # Ghost body at setpoint (dashed reference pose)
    if trail_ghost:
        Rdes = rot(theta_des)
        ghost_verts = body_to_world(verts0, theta_des)
        ghost = plt.Polygon(
            ghost_verts, closed=True, fill=False, edgecolor="#888",
            lw=1.2, ls="--", zorder=2,
        )
        ax.add_patch(ghost)
        wc_des = Rdes @ np.array([0.0, l])
        ghost_wheel = Circle(
            (wc_des[0], wc_des[1]), r_wheel, fill=False, edgecolor="#888",
            lw=1.0, ls="--", zorder=2,
        )
        ax.add_patch(ghost_wheel)

    # Live cube body
    body = plt.Polygon(
        verts0, closed=True, facecolor="#d8e2ec", edgecolor="#1a1a1a",
        lw=2.0, alpha=0.95, zorder=4,
    )
    ax.add_patch(body)

    # Structure lines
    diag_line, = ax.plot([], [], color="#4a5560", lw=1.0, zorder=5)
    cross_line, = ax.plot([], [], color="#4a5560", lw=0.8, ls=":", zorder=5)

    # Reaction wheel (rim + hub + spokes)
    wheel_rim = Circle((0, l), r_wheel, facecolor="#f0f0f0", edgecolor="#111",
                       lw=2.0, zorder=6)
    wheel_hub = Circle((0, l), 0.006, facecolor="#111", zorder=8)
    ax.add_patch(wheel_rim)
    ax.add_patch(wheel_hub)
    n_spokes = 6
    spoke_lines = [
        ax.plot([], [], color="#222", lw=1.3, zorder=7)[0] for _ in range(n_spokes)
    ]

    # Pivot joint
    ax.add_patch(Circle((0, 0), 0.008, facecolor="#111", zorder=10))
    ax.plot([-0.018, 0, 0.018], [0.0, 0.012, 0.0], color="#111", lw=1.2, zorder=9)

    # CoM + wheel-centre as Circle patches (track body without marker lag)
    com_dot = Circle((0.0, lb), 0.0045, facecolor="#c0392b", edgecolor="#8b1e14", lw=0.8, zorder=9)
    wheel_center_dot = Circle((0.0, l), 0.004, facecolor="#1f6feb", edgecolor="#0b3d91", lw=0.8, zorder=9)
    ax.add_patch(com_dot)
    ax.add_patch(wheel_center_dot)

    # Angle arc between vertical and body diagonal
    # Convention: θb = 0 upright; θb > 0 rotates body CCW (top leans to -x)
    vert_ray, = ax.plot([0, 0], [0, 0.055], color="#888", lw=1.2, ls="--", zorder=8)
    angle_arc_line, = ax.plot([], [], color="#c0392b", lw=1.8, zorder=8)
    angle_label = ax.text(0, 0, "", color="#c0392b", fontsize=9, ha="center", va="center", zorder=11)

    # Dimension line for wheel axis distance l (lb shown by red CoM marker)
    dim_l_line, = ax.plot([], [], color="#1f6feb", lw=1.0, zorder=3)
    label_l = ax.text(0, 0, "", color="#1f6feb", fontsize=9, zorder=11)

    info = ax.text(
        0.02, 0.97, "", transform=ax.transAxes, va="top",
        fontsize=10, family="monospace",
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#bbb", alpha=0.9),
        zorder=20,
    )

    ax.set_xlim(-0.26, 0.28)
    ax.set_ylim(-0.04, 0.22)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(title, fontsize=13, pad=8)
    ax.grid(True, alpha=0.18)
    ax.text(
        0.98, 0.03,
        "solid = current pose   dashed = upright reference",
        transform=ax.transAxes, ha="right", fontsize=8, color="#555",
    )

    def update(frame):
        tb = th_b[frame]
        tw = th_w[frame]
        R = rot(tb)

        verts = body_to_world(verts0, tb)
        body.set_xy(verts)

        d1 = body_to_world(np.array([[0.0, 0.0], [0.0, diag]]), tb)
        d2 = body_to_world(np.array([[-half, half], [half, half]]), tb)
        diag_line.set_data(d1[:, 0], d1[:, 1])
        cross_line.set_data(d2[:, 0], d2[:, 1])

        wc = R @ np.array([0.0, l])
        wheel_rim.center = (wc[0], wc[1])
        wheel_hub.center = (wc[0], wc[1])
        wheel_center_dot.center = (float(wc[0]), float(wc[1]))

        com = R @ np.array([0.0, lb])
        com_dot.center = (float(com[0]), float(com[1]))

        for i, sp in enumerate(spoke_lines):
            phi = tw + tb + i * (2 * np.pi / n_spokes)
            tip = wc + r_wheel * 0.92 * np.array([np.cos(phi), np.sin(phi)])
            sp.set_data([wc[0], tip[0]], [wc[1], tip[1]])

        # Body diagonal world angle from +x: π/2 + θb
        # (matches R_ccw @ [0, y] = (-y sin θb, y cos θb))
        arc_r = 0.042
        n_arc = max(2, int(abs(tb) * 180 / np.pi) + 2)
        arc_angs = np.linspace(np.pi / 2, np.pi / 2 + tb, n_arc)
        angle_arc_line.set_data(arc_r * np.cos(arc_angs), arc_r * np.sin(arc_angs))

        # Label sitting on the mid-arc
        mid = np.pi / 2 + 0.5 * tb
        angle_label.set_position((1.35 * arc_r * np.cos(mid), 1.35 * arc_r * np.sin(mid)))
        angle_label.set_text(rf"$\theta_b$={tb * 180 / np.pi:.1f}°")

        if show_labels:
            n_hat = R @ np.array([0.018, 0.0])
            p0 = n_hat
            p_l = wc + n_hat
            dim_l_line.set_data([p0[0], p_l[0]], [p0[1], p_l[1]])
            mid_l = 0.5 * (p0 + p_l) + n_hat
            label_l.set_position((mid_l[0], mid_l[1]))
            label_l.set_text(r"$l$")

        info.set_text(
            f"t = {t[idx[frame]]:5.2f} s\n"
            f"θb = {tb * 180 / np.pi:6.2f}°\n"
            f"ωb = {wb[frame]:6.2f} rad/s\n"
            f"ωw = {ww[frame]:6.1f} rad/s"
        )
        return (
            body, diag_line, cross_line, wheel_rim, wheel_hub,
            com_dot, wheel_center_dot, angle_arc_line, angle_label,
            info, *spoke_lines,
        )

    anim = FuncAnimation(fig, update, frames=n_frames, interval=interval, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())


# Realistic animation of the LQR balancing run
animate_cubli(t_lqr, X_lqr, title="Cubli LQR balancing (schematic geometry)")


## 7b. Interactive simulation + Cubli animation

- **LQR:** desired state fixed at **upright** ($\theta_{des}=0$). Change **initial** conditions only.
- **PID:** setpoint $\theta_{des}$ is editable; initial conditions too.

Click **Run simulation** to update the state plots and the Cubli animation.

Run all cells above this one first (through LQR + `animate_cubli`).


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

_missing = [
    n for n in ("K_lqr", "simulate", "accelerations", "Load", "pid", "animate_cubli")
    if n not in globals()
]
if _missing:
    raise RuntimeError("Run earlier cells first. Missing: " + ", ".join(_missing))

_icfg = {"theta_des": 0.0, "controller": "LQR", "disturbance": False}


def _pid_torque(t, X):
    kp, ki, kd = 20.0, 10.0, 15.0
    e = X[0] - _icfg["theta_des"]
    de = X[2]
    dt = t - pid.tp
    if dt > 0:
        pid.integral += ki * e * dt
    pid.tp = t
    return kp * e + pid.integral + kd * de


def _lqr_torque(X):
    # Classic upright LQR only: u = -K X  (θ_des fixed at 0)
    return float(-K_lqr @ X)


def _plant(t, X, nonlinear=True):
    load = Load(t) if _icfg["disturbance"] else 0.0
    x1, x3, x4 = X[0], X[2] + load, X[3]
    Torque = _pid_torque(t, X) if _icfg["controller"] == "PID" else _lqr_torque(X)
    ddb, ddw = accelerations(x1, x3, x4, Torque, nonlinear=nonlinear)
    return [x3, x4, ddb, ddw]


def run_interactive(
    theta_des_deg=0.0,
    theta0_deg=22.5,
    omega_b0=0.0,
    omega_w0=0.0,
    controller="LQR",
    plant="nonlinear",
    t_final=15.0,
    disturbance=False,
    n_frames=100,
):
    # LQR always balances to upright
    if controller == "LQR":
        theta_des_deg = 0.0

    _icfg["theta_des"] = np.deg2rad(float(theta_des_deg))
    _icfg["controller"] = controller
    _icfg["disturbance"] = bool(disturbance)

    x0 = np.array([np.deg2rad(float(theta0_deg)), 0.0, float(omega_b0), float(omega_w0)])
    nonlinear = plant == "nonlinear"
    fun = lambda t, X, nl=nonlinear: _plant(t, X, nonlinear=nl)

    t, X = simulate(fun, (0.0, float(t_final)), x0, n_eval=1500)

    fig, axs = plt.subplots(3, 1, figsize=(9, 6.2), sharex=True)
    fig.suptitle(
        f"{controller} | {plant} | θ_des={theta_des_deg:.1f}° | θ0={theta0_deg:.1f}°"
    )
    axs[0].plot(t, X[:, 0] * 180 / np.pi, label=r"$\theta_b$")
    axs[0].axhline(theta_des_deg, color="C3", ls="--", lw=1.2, label=r"$\theta_{des}$")
    axs[0].set_ylabel(r"$\theta_b$ (deg)")
    axs[0].legend(loc="upper right")
    axs[1].plot(t, X[:, 2])
    axs[1].set_ylabel(r"$\dot\theta_b$ (rad/s)")
    axs[2].plot(t, X[:, 3])
    axs[2].set_ylabel(r"$\dot\theta_w$ (rad/s)")
    axs[2].set_xlabel("time (s)")
    for ax in axs:
        ax.grid(True, alpha=0.3)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

    display(
        animate_cubli(
            t,
            X,
            title=(
                f"Cubli animation | {controller}, "
                f"θ_des={theta_des_deg:.1f}°, θ0={theta0_deg:.1f}°"
            ),
            n_frames=int(n_frames),
            interval=35,
            show_labels=True,
            trail_ghost=True,
            theta_des=_icfg["theta_des"],
        )
    )


style = {"description_width": "130px"}
layout = widgets.Layout(width="460px")

w_ctrl = widgets.ToggleButtons(
    options=["LQR", "PID"], value="LQR", description="Controller", style=style
)
w_theta_des = widgets.FloatSlider(
    value=0.0, min=-30.0, max=30.0, step=0.5,
    description="Setpoint θb (°) [fixed]", style=style, layout=layout,
    continuous_update=False, disabled=True,
)
w_theta0 = widgets.FloatSlider(
    value=22.5, min=-45.0, max=45.0, step=0.5,
    description="Initial θb (°)", style=style, layout=layout, continuous_update=False,
)
w_wb0 = widgets.FloatSlider(
    value=0.0, min=-5.0, max=5.0, step=0.1,
    description="Initial ωb", style=style, layout=layout, continuous_update=False,
)
w_ww0 = widgets.FloatSlider(
    value=0.0, min=-100.0, max=100.0, step=1.0,
    description="Initial ωw", style=style, layout=layout, continuous_update=False,
)
w_plant = widgets.ToggleButtons(
    options=["nonlinear", "linear"], value="nonlinear", description="Plant", style=style
)
w_tfinal = widgets.FloatSlider(
    value=15.0, min=5.0, max=40.0, step=1.0,
    description="Sim time (s)", style=style, layout=layout, continuous_update=False,
)
w_dist = widgets.Checkbox(value=False, description="Disturbance pulse at t=15–16 s")
w_frames = widgets.IntSlider(
    value=100, min=40, max=200, step=10,
    description="Anim frames", style=style, layout=layout, continuous_update=False,
)
btn = widgets.Button(description="Run simulation", button_style="success", icon="play")
status = widgets.HTML(value="<i>Ready — change values, then click Run.</i>")
hint = widgets.HTML(
    value=(
        "<p style='color:#555;margin:4px 0 8px 0'>"
        "<b>LQR:</b> desired state fixed at upright <b>0°</b> "
        "(classic $u=-KX$). Change initial conditions only.<br>"
        "<b>PID:</b> setpoint θ<sub>des</sub> is editable."
        "</p>"
    )
)
out = widgets.Output()


def _sync_setpoint_for_controller(change=None):
    if w_ctrl.value == "LQR":
        w_theta_des.value = 0.0
        w_theta_des.disabled = True
        w_theta_des.description = "Setpoint θb (°) [fixed]"
    else:
        w_theta_des.disabled = False
        w_theta_des.description = "Setpoint θb (°)"


w_ctrl.observe(_sync_setpoint_for_controller, names="value")
_sync_setpoint_for_controller()


def _on_run(_=None):
    status.value = "<b style='color:#0a7'>Running…</b> (animation may take a few seconds)"
    with out:
        clear_output(wait=True)
        try:
            run_interactive(
                theta_des_deg=w_theta_des.value,
                theta0_deg=w_theta0.value,
                omega_b0=w_wb0.value,
                omega_w0=w_ww0.value,
                controller=w_ctrl.value,
                plant=w_plant.value,
                t_final=w_tfinal.value,
                disturbance=w_dist.value,
                n_frames=w_frames.value,
            )
            des = 0.0 if w_ctrl.value == "LQR" else w_theta_des.value
            status.value = (
                f"<b style='color:#0a7'>Done.</b> "
                f"{w_ctrl.value}: θ_des={des:.1f}°, θ0={w_theta0.value:.1f}°, "
                f"{w_plant.value}"
            )
        except Exception as exc:
            status.value = f"<b style='color:#c00'>Error:</b> {exc}"
            raise


btn.on_click(_on_run)

ui = widgets.VBox([
    widgets.HTML(
        "<h3 style='margin:0 0 6px 0'>Interactive Cubli control + animation</h3>"
        "<p style='margin:0 0 8px 0;color:#444'>Adjust controls, then press "
        "<b>Run simulation</b> for plots and Cubli animation.</p>"
    ),
    w_ctrl,
    hint,
    w_theta_des,
    w_theta0,
    w_wb0,
    w_ww0,
    w_plant,
    w_tfinal,
    w_dist,
    w_frames,
    widgets.HBox([btn, status]),
    out,
])

display(ui)
_on_run()


## 8. Optional: 3-state model from `topic_8.mlx`

That script used $X = [\theta_b,\; \dot\theta_b,\; \dot\theta_w]^T$ (no wheel angle state) with a larger initial tilt and wheel spin.

In [ ]:
def sysdyn_3(t, X):
    x1 = X[0] + disturbance_angle(t)
    x2, x3 = X[1], X[2]
    Torque = pid_control(t, X, use_4state=False)
    ddb, ddw = accelerations(x1, x2, x3, Torque, nonlinear=True)
    return [x2, ddb, ddw]


def lin_sysdyn_3(t, X):
    x1, x2, x3 = X
    Torque = pid_control(t, X, use_4state=False)
    ddb, ddw = accelerations(x1, x2, x3, Torque, nonlinear=False)
    return [x2, ddb, ddw]


def sysdyn_lqr_3(t, X):
    x1, x2, x3 = X
    Torque = float(-K_lqr3 @ X)
    ddb, ddw = accelerations(x1, x2, x3, Torque, nonlinear=False)
    return [x2, ddb, ddw]


A3 = np.array([
    [0, 1, 0],
    [(Mb * lb + Mw * l) * g / denom, -Cb / denom, Cw / denom],
    [-(Mb * lb + Mw * l) * g / denom, Cb / denom, (Ib + Iw + Mw * l * l) * (-Cw) / (Iw * denom)],
])
B3 = np.array([
    [0.0],
    [-1.0 / denom],
    [(Ib + Iw + Mw * l * l) / (Iw * denom)],
])
Q3 = np.diag([2.0, 5.0, 100.0])
R3 = np.array([[500.0]])
P3 = solve_continuous_are(A3, B3, Q3, R3)
K_lqr3 = (np.linalg.solve(R3, B3.T @ P3)).ravel()
print("3-state LQR gain K =", K_lqr3)

t_span3 = (0.0, 20.0)
x0_3 = np.array([np.pi / 4, 0.0, 70.0])  # topic_8 initial conditions

t3, X3 = simulate(sysdyn_3, t_span3, x0_3)
fig, axs = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
fig.suptitle("PID — nonlinear 3-state model (topic_8)")
labels = [r"$\theta_b$ (rad)", r"$\dot\theta_b$", r"$\dot\theta_w$"]
for i, ax in enumerate(axs):
    ax.plot(t3, X3[:, i])
    ax.set_ylabel(labels[i])
    ax.grid(True, alpha=0.3)
axs[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

t3l, X3l = simulate(sysdyn_lqr_3, t_span3, x0_3)
fig, axs = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
fig.suptitle("LQR — linear 3-state model (topic_8)")
for i, ax in enumerate(axs):
    ax.plot(t3l, X3l[:, i])
    ax.set_ylabel(labels[i])
    ax.grid(True, alpha=0.3)
axs[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

## Summary

- **Plant:** planar Cubli / reaction-wheel inverted pendulum (nonlinear + linearized).
- **Controllers applied:**
  1. **PID** — $K_p=20$, $K_i=10$, $K_d=15$ on $\theta_b$ (linear and nonlinear models).
  2. **LQR** — classic upright state feedback $u=-KX$ with desired state $\theta_b=0$.
- Disturbance pulse near $t=15$–$16$ s tests disturbance rejection.
- Interactive cell: LQR setpoint locked at 0°; PID setpoint editable; Cubli animation included.
